In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../datasets/7 churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# **Data Pre-processing**

In [3]:
df.shape

(7043, 21)

In [4]:
df.isnull().sum()  # Count missing values in each column

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

- **KEEPING IMPORTANT COLUMNS**

In [5]:
columns_Keeping = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'Contract', 'TotalCharges', 'Churn']

df = df[columns_Keeping]  # Filter the dataframe to the selected columns
df.head()  # Preview the cleaned dataframe

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,Contract,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,Month-to-month,29.85,No
1,Male,0,No,No,34,Yes,No,One year,1889.5,No
2,Male,0,No,No,2,Yes,No,Month-to-month,108.15,Yes
3,Male,0,No,No,45,No,No phone service,One year,1840.75,No
4,Female,0,No,No,2,Yes,No,Month-to-month,151.65,Yes


# **ENCODE Binary Variables (yes/no columns)**

In [6]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'Contract', 'Churn']  # Define columns that need encoding

for cols in categorical_cols:  # Encode each selected column
    df[cols] = label_encoder.fit_transform(df[cols].astype(str))  # Convert values to numeric labels

df  # Display the dataframe after encoding

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,Contract,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,29.85,0
1,1,0,0,0,34,1,0,1,1889.5,0
2,1,0,0,0,2,1,0,0,108.15,1
3,1,0,0,0,45,0,1,1,1840.75,0
4,0,0,0,0,2,1,0,0,151.65,1
...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,2,1,1990.5,0
7039,0,0,1,1,72,1,2,1,7362.9,0
7040,0,0,1,1,11,0,1,0,346.45,0
7041,1,1,1,0,4,1,2,0,306.6,1


# **Train Test Split**

In [7]:
X = df.drop('Churn', axis=1)  # Separate features from the target variable
y = df['Churn']  # Store the target column

In [8]:
from sklearn.model_selection import train_test_split  # Import train-test split function
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # Split data into train and test sets

In [9]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 2142 to 860
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   gender         5634 non-null   int64
 1   SeniorCitizen  5634 non-null   int64
 2   Partner        5634 non-null   int64
 3   Dependents     5634 non-null   int64
 4   tenure         5634 non-null   int64
 5   PhoneService   5634 non-null   int64
 6   MultipleLines  5634 non-null   int64
 7   Contract       5634 non-null   int64
 8   TotalCharges   5634 non-null   str  
dtypes: int64(8), str(1)
memory usage: 473.2 KB


# **Replacing Missing Values in Total charges column with mean of the column**

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # Recreate train/test split for the prepared data

X_train['TotalCharges'] = pd.to_numeric(X_train['TotalCharges'], errors='coerce')  # Convert TotalCharges to numeric, treating invalid values as missing
X_test['TotalCharges'] = pd.to_numeric(X_test['TotalCharges'], errors='coerce')  # Do the same for the test set

X_train['TotalCharges'] = X_train['TotalCharges'].fillna(X_train['TotalCharges'].mean())  # Fill missing values with the mean of the training set
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(X_test['TotalCharges'].mean())  # Fill missing values in the test set using the same mean

# **LOGISTIC REGRESSION**

In [11]:
from sklearn.linear_model import LogisticRegression  # Import logistic regression model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Import evaluation metrics

lg = LogisticRegression(max_iter=1000)  # Create the logistic regression model
lg.fit(X_train, y_train)  # Train the model on the training data
y_pred = lg.predict(X_test)  # Predict the target for the test data

accuracy = accuracy_score(y_test, y_pred)  # Calculate model accuracy
conf_matrix = confusion_matrix(y_test, y_pred)  # Calculate confusion matrix

print('Accuracy:', accuracy)  # Print the accuracy score
print('Confusion Matrix:')  # Print the confusion matrix header
print(conf_matrix)  # Display the confusion matrix
print('\nClassification Report:')  # Print the classification report header
print(classification_report(y_test, y_pred))  # Display precision, recall, and F1-score

Accuracy: 0.7757274662881476
Confusion Matrix:
[[947  89]
 [227 146]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.86      1036
           1       0.62      0.39      0.48       373

    accuracy                           0.78      1409
   macro avg       0.71      0.65      0.67      1409
weighted avg       0.76      0.78      0.76      1409



- **SAVE MODEL**

In [12]:
import pickle
pickle.dump(lg, open('logistc_model.pkl', 'wb'))